# District-Species Occupancy Analysis (Thinned)

Presence-based district comparison for each species using occupancy, not individual counts.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='paper')
RANDOM_SEED = 42


In [ ]:
p = next((x for x in [Path('file6.csv'), Path('../file6.csv'), Path('../../file6.csv')] if x.exists()), None)
if p is None:
    raise FileNotFoundError('file6.csv not found')
cols = ['stateProvince','verbatimScientificName','decimalLatitude','decimalLongitude','eventDate']
df = pd.read_csv(p, usecols=cols, low_memory=False)
df['eventDate'] = pd.to_datetime(df['eventDate'], errors='coerce')
for c in ['decimalLatitude','decimalLongitude']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = df.dropna(subset=cols)
df['stateProvince'] = df['stateProvince'].astype(str).str.strip()
df['verbatimScientificName'] = df['verbatimScientificName'].astype(str).str.strip()
print(f'Rows: {len(df):,} | Districts: {df.stateProvince.nunique()} | Species: {df.verbatimScientificName.nunique()}')


In [ ]:
lon0 = float(df['decimalLongitude'].median())
lat0 = float(df['decimalLatitude'].median())
lat_rad = np.radians(df['decimalLatitude'].to_numpy())
df['x_km'] = (df['decimalLongitude'].to_numpy() - lon0) * 111.320 * np.cos(lat_rad)
df['y_km'] = (df['decimalLatitude'].to_numpy() - lat0) * 110.574
df['grid_x'] = np.floor(df['x_km'] / 5.0).astype(int)
df['grid_y'] = np.floor(df['y_km'] / 5.0).astype(int)
df['cell_id'] = df['stateProvince'].astype(str) + '|' + df['grid_x'].astype(str) + '_' + df['grid_y'].astype(str)
df['_key'] = df['cell_id'] + '|' + df['verbatimScientificName']
rng = np.random.default_rng(RANDOM_SEED)
df['_r'] = rng.random(len(df))
df_thin = df.sort_values('_r').groupby('_key', as_index=False).head(1).drop(columns=['_key','_r'])
print(f'Thinned rows: {len(df_thin):,}')


In [ ]:
cells_per_district = df_thin.groupby('stateProvince')['cell_id'].nunique().rename('n_cells_total')
occ_counts = df_thin.groupby(['stateProvince','verbatimScientificName'])['cell_id'].nunique().rename('n_cells_occupied').reset_index()
occ = occ_counts.merge(cells_per_district.reset_index(), on='stateProvince', how='left')
occ['occupancy_rate'] = occ['n_cells_occupied'] / occ['n_cells_total']
occ = occ.sort_values(['verbatimScientificName','occupancy_rate'], ascending=[True,False])
occ.head(20)


In [ ]:
top_species = occ.groupby('verbatimScientificName')['n_cells_occupied'].sum().nlargest(20).index
heat = occ[occ['verbatimScientificName'].isin(top_species)].pivot(index='stateProvince', columns='verbatimScientificName', values='occupancy_rate').fillna(0)
plt.figure(figsize=(13,8))
sns.heatmap(heat, cmap='YlGnBu', linewidths=0.3)
plt.title('District-species occupancy rate (top 20 species by occupied cells)')
plt.tight_layout()
plt.show()


In [ ]:
top_rank = occ.sort_values('occupancy_rate', ascending=False).groupby('verbatimScientificName').head(3)
top_rank.head(30)
